# Machine Learning Pipeline

In [ ]:
import json
import keras
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler


## 1. Parameter Global


In [ ]:
RANDOM_STATE = 42

TICKER = "BBRI.JK"
TARGET_COL = "close"

TIME_STEPS = [10, 20, 30]
OPTIMIZERS = ["SGD", "Adam", "RMSprop"]
BATCH_SIZES = [8, 16, 32]
LEARNING_RATES = [0.01, 0.001, 0.0001]
MODELS = ["LSTM", "GRU"]

EPOCHS = 50
TRAIN_SIZE = 0.80

In [ ]:
DATA_PATH = Path(f"artifact/data/{TICKER}.parquet") if Path(f"artifact/data/{TICKER}.parquet").exists() else Path(f"../artifact/data/{TICKER}.parquet")
OUTPUT_DIR = Path("artifact/model") if Path("artifact").exists() else Path("../artifact/model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
keras.utils.set_random_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Data Understanding


### 2.1. Memuat Data


In [ ]:
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Periode Observasi : {df['date'].min().date()} s/d {df['date'].max().date()}")
print(f"Total Baris Data  : {len(df)}")

### 2.2. Pengecekan Gap Tanggal


In [ ]:
full_date_range = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
missing_dates = full_date_range.difference(df["date"])

print(f"Jumlah Gap Tanggal : {len(missing_dates)}")

### 2.3. Visualisasi Harga Historis


In [ ]:
plt.figure(figsize=(13, 5))
sns.lineplot(data=df, x="date", y=TARGET_COL)
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.title(f"Pergerakan Harga Historis - {TICKER}")
plt.tight_layout()
plt.show()

## 3. Data Preparation


### 3.2. Pembagian Data


In [ ]:
train_len = int(len(df) * TRAIN_SIZE)
train_df = df.iloc[:train_len].copy().reset_index(drop=True)
test_df = df.iloc[train_len:].copy().reset_index(drop=True)

print(f"Jumlah Data Latih : {len(train_df)}")
print(f"Jumlah Data Uji   : {len(test_df)}")

### 3.3. Normalisasi Skala


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[[TARGET_COL]].values)
test_scaled = scaler.transform(test_df[[TARGET_COL]].values)

In [ ]:
print(f"Min Scaler (Train)  : {scaler.data_min_[0]:.2f}")
print(f"Max Scaler (Train)  : {scaler.data_max_[0]:.2f}")

In [ ]:
def create_windowed_dataset(train_arr, test_arr, time_steps, batch_size):
    ds_train = keras.utils.timeseries_dataset_from_array(
        data=train_arr[:-1],
        targets=train_arr[time_steps:, 0],
        sequence_length=time_steps,
        batch_size=batch_size,
        shuffle=True,
        seed=RANDOM_STATE
    )

    test_inputs = np.concatenate([train_arr[-time_steps:], test_arr], axis=0)

    ds_test = keras.utils.timeseries_dataset_from_array(
        data=test_inputs[:-1],
        targets=test_inputs[time_steps:, 0],
        sequence_length=time_steps,
        batch_size=batch_size,
        shuffle=False
    )

    return ds_train, ds_test

## 4. Model & Evaluasi


### 4.1. Arsitektur Model

In [ ]:
def build_model(model_type, time_steps, opt_name, lr):
    keras.backend.clear_session()

    # Konfigurasi algoritma optimasi dan learning rate
    if opt_name == "SGD":
        opt = keras.optimizers.SGD(learning_rate=lr)
    elif opt_name == "Adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    elif opt_name == "RMSprop":
        opt = keras.optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError(f"Optimizer {opt_name} tidak didukung.")

    # Inisialisasi arsitektur sequential
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(time_steps, 1)))

    # Layer recurrent tunggal
    if model_type == "LSTM":
        model.add(keras.layers.LSTM(units=50, return_sequences=False))
    elif model_type == "GRU":
        model.add(keras.layers.GRU(units=50, return_sequences=False))
    else:
        raise ValueError(f"Tipe model {model_type} tidak valid.")

    # Layer output regresi linier tunggal
    model.add(keras.layers.Dense(units=1))
    model.compile(optimizer=opt, loss="mean_squared_error")

    return model

In [ ]:
def calculate_metrics(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape
    }


### 4.2. Eksperimen Model

In [ ]:
def execute_trial(trial_params, train_scaled, test_scaled, y_actual, scaler, epochs):
    ts, opt, bs, lr, m_type = trial_params

    # Siapkan windowed dataset untuk trial terkait
    ds_train, ds_test = create_windowed_dataset(
        train_arr=train_scaled,
        test_arr=test_scaled,
        time_steps=ts,
        batch_size=bs
    )

    # Latih model secara statis sejumlah epoch
    model = build_model(model_type=m_type, time_steps=ts, opt_name=opt, lr=lr)
    model.fit(ds_train, epochs=epochs, shuffle=False, verbose=0)

    # Inferensi dan inversi skala target
    preds_scaled = model.predict(ds_test, verbose=0)
    preds_inv = scaler.inverse_transform(preds_scaled).flatten()

    metrics = calculate_metrics(y_true=y_actual, y_pred=preds_inv)

    record = {
        "model": m_type,
        "time_steps": ts,
        "optimizer": opt,
        "batch_size": bs,
        "learning_rate": lr,
        **metrics
    }

    return record, model

In [ ]:
param_combinations = list(itertools.product(
    TIME_STEPS,
    OPTIMIZERS,
    BATCH_SIZES,
    LEARNING_RATES,
    MODELS
))

results = []
y_actual = test_df[TARGET_COL].values
progress_bar = tqdm(param_combinations, desc=f"Grid Search", unit="trial")

# Variabel rekor RMSE terendah per tipe model
best_rmse_tracker = {}

for m_name in MODELS:
    best_rmse_tracker[m_name] = float("inf")

# Perulangan grid search dan penyimpanan model otomatis
for params in progress_bar:
    record, trained_model = execute_trial(
        trial_params=params,
        train_scaled=train_scaled,
        test_scaled=test_scaled,
        y_actual=y_actual,
        scaler=scaler,
        epochs=EPOCHS
    )

    results.append(record)

    current_model = record["model"]
    current_rmse = record["RMSE"]

    # Simpan model terbaik
    if current_rmse < best_rmse_tracker[current_model]:
        best_rmse_tracker[current_model] = current_rmse
        trained_model.save(OUTPUT_DIR / f"{TICKER}_{current_model}.keras")

    progress_bar.set_postfix({
        "Model": current_model,
        "RMSE": f"{current_rmse:.2f}"
    })

# Simpan riwayat lengkap eksperimen
results_df = pd.DataFrame(results)
results_df.to_parquet(OUTPUT_DIR / f"{TICKER}_hyperparameter.parquet", index=False)
results_df

### 4.3. Penentuan Paramter Terbaik

In [ ]:
grouped_by_model = results_df.groupby("model")
best_indices_series = grouped_by_model["RMSE"].idxmin()
best_indices = best_indices_series.values

best_configs_df = results_df.loc[best_indices].reset_index(drop=True)

best_configs = {}
for _, row in best_configs_df.iterrows():
    best_configs[row["model"]] = row.to_dict()

summary_best_df = best_configs_df.copy()
summary_best_df

### 4.4. Inferensi Model


In [ ]:
for model_name, cfg in best_configs.items():
    ts = int(cfg["time_steps"])
    bs = int(cfg["batch_size"])

    # Muat model terbaik
    model_path = OUTPUT_DIR / f"{TICKER}_{model_name}.keras"
    loaded_model = keras.models.load_model(model_path)

    # Siapkan windowed dataset
    _, ds_test = create_windowed_dataset(train_scaled, test_scaled, ts, bs)

    # Lakukan prediksi pada data uji
    pred_test = scaler.inverse_transform(loaded_model.predict(ds_test, verbose=0)).flatten()

    # Buat plot
    plt.figure(figsize=(13, 5))

    plt.plot(train_df["date"].iloc[ts:], train_df[TARGET_COL].iloc[ts:], label="Train Actual")
    plt.plot(test_df["date"], test_df[TARGET_COL], label="Test Actual")
    plt.plot(test_df["date"], pred_test, label=f"{model_name} Test Prediction")

    plt.title(f"Hasil Inferensi Model - {TICKER}")
    plt.xlabel("Tanggal")
    plt.ylabel("Harga (Rp)")
    plt.legend()
    plt.tight_layout()
    plt.show()